In [ ]:
import pandas as pd
import numpy as np
import re
from tqdm import tqdm
import os
import importlib
import time
import itertools
import datetime
import ipynbname

from kfirlib.comparisons import *
from kfirlib.util import *
from kfirlib import pubmed, gpt

In [ ]:
DIR = r'C:\Users\israe\OneDrive\Documents\post doc\kfir\Jumalah\tables\fromJamalah9_6'
IN_PATH = os.path.join(DIR, '86_with_inclusion.xlsx')
OUT_PATH = os.path.join(DIR, '86_again.xlsx')
LOG_PATH = os.path.join(DIR, 'log.txt')

In [ ]:
orig_df = pd.read_excel(IN_PATH)
orig_df.columns

In [ ]:
is_missing(orig_df.abstract).sum()

In [ ]:
orig_df.publication_date.head()

In [ ]:
pd.to_datetime(orig_df.publication_date.head()).dt.year

In [ ]:
orig_df['year'] = pd.to_datetime(orig_df.publication_date).dt.year
orig_df.head()

In [ ]:
eligibility_abstract = gpt.check_eligibility(orig_df, criteria=gpt.JUMALAH_CRITERIA)
eligibility_abstract.columns

In [ ]:
eligibility_abstract.rename(columns={'reasoning': 'reasoning_title_abstract', 'decision': 'decision_title_abstract'}, inplace=True)
eligibility_abstract.columns

In [ ]:
no_abstract = eligibility_abstract.copy()
no_abstract.loc['abstract', :] = np.nan
no_abstract_res = gpt.check_eligibility(no_abstract, criteria=gpt.JUMALAH_CRITERIA, batch_size=100)

In [ ]:
part = no_abstract_res[['reasoning', 'decision']].copy()
part.rename(columns={'reasoning': 'reasoning_title', 'decision': 'decision_title'}, inplace=True)
eligibility_all = pd.concat([eligibility_abstract, part], axis=1)
eligibility_all.head()

In [ ]:
for i, col in enumerate(eligibility_all.columns):
    print(i, col)

In [ ]:
col_inds = list(range(len(eligibility_all.columns)))
col_inds.remove(17)
col_inds.remove(18)
eligibility_all = eligibility_all.iloc[:, col_inds]
eligibility_all.columns

In [ ]:
eligibility_all.decision_title.isna().sum(), eligibility_all.decision_title_abstract.isna().sum()

In [ ]:
eligibility_all.decision_title.mean(), eligibility_all.decision_title_abstract.mean(), (eligibility_all.decision_title != eligibility_all.decision_title_abstract).mean()

In [ ]:
eligibility_all.to_excel(OUT_PATH, index=False)

## Confusion matrix

In [ ]:
#eligibility_all = pd.read_excel(OUT_PATH)
eligibility_all.columns

In [ ]:
decision = eligibility_all.decision_title
gt = eligibility_all.in_495 == 1

In [ ]:
def confusion_matrix(decision, gt):
    df = pd.DataFrame({'decision': decision, 'truth': gt})
    df['val'] = 1
    res = df.groupby(['decision', 'truth']).sum().reset_index(drop=False)
    return res.pivot(index='decision', columns='truth', values='val')    

In [ ]:
log_file = open(LOG_PATH, 'at')

In [ ]:
log_file.write(f'\n==================== {datetime.datetime.now().strftime("%Y-%m-%d %H:%M")} ====================\n\n')
log_file.write(f'{ipynbname.path()}\n')
log_file.write(f'Input: {IN_PATH}\n')
log_file.write(f'Output: {OUT_PATH}\n')

In [ ]:
log_file.write(f'\nSystem prompt:\n\n{gpt.SYSTEM_PROMPT}\n\n')
log_file.write(f'\nCriteria:\n\n{gpt.JUMALAH_CRITERIA}\n\n')

Title

In [ ]:
confusion = confusion_matrix(decision=eligibility_all.decision_title, gt=(eligibility_all.in_495 == 1))
confusion

In [ ]:
log_file.write(f'\nConfusion for titles:\n{confusion}\n\n')

Title and abstract

In [ ]:
# confusion = confusion_matrix(decision=eligibility_all['decision_title_abstract'], gt=eligibility_all.in_86)
# confusion

In [ ]:
# log_file.write(f'\nConfusion for titles:\n{confusion}\n\n')

Title, abstract, AND

In [ ]:
confusion = confusion_matrix(decision=eligibility_all.decision_title_abstract & eligibility_all.decision_title, gt=eligibility_all.in_final_30)
confusion

In [ ]:
log_file.write(f'\nConfusion for AND:\n{confusion}\n\n')

In [ ]:
log_file.close()